# Spektrale Dichte $\rho_{\mathrm{spec}}(M)$ — fünf Faktoren (SageMath)

Dieses Notebook setzt die **fünf Schritte** in einen ausführbaren SageMath-Code um:

1. $\rho_{\mathrm{slot}}$ — besetzte Primslots auf der reduzierten Hülle $H_q(R)$
2. $B_{\mathrm{EABC}}$ — EABC-Balance (Modulo 12)
3. $S_{\mathrm{sym}}$ — Symmetrie um $M$ (Paare $\pm r$)
4. $\Theta_{\mathrm{rank}}$ — Rangspannung aus der normierten Übergangsmatrix (Singulärwerte)
5. $\eta_{\mathrm{ent}}$ — normierte spektrale Entropie

Finale Kombination (wie im Gespräch vereinbart):

$$\rho_{\mathrm{spec}}(M) = \rho_{\mathrm{slot}} \cdot B_{\mathrm{EABC}} \cdot S_{\mathrm{sym}} \cdot \Theta_{\mathrm{rank}} \cdot \eta_{\mathrm{ent}}$$

**Wichtig:** Der Code ist eine **direkte Übersetzung** der Herleitung aus dem Gespräch, **nicht** aus den erwähnten Quelltexten. Bitte **eigenständig prüfen** (Randfälle, Parameter $q,R,\beta_E,\beta_S$, Definition von $H_q(R)$).

**Kernel:** SageMath wählen (Jupyter-Name meist `sagemath-10.9`, Anzeige „SageMath 10.9“). **Nicht** den reinen Python-3-Kernel aus conda/pyenv — sonst fehlt `sage.all`.

**Ausführen:** Eine `.ipynb`-Datei ist **kein** Shell-Kommando (nur den Namen einzutippen startet nichts). Stattdessen: Notebook in Cursor öffnen und Kernel **SageMath** wählen, alle Zellen ausführen — oder im Terminal z. B. `jupyter run Spektrale_Dichte_5Faktoren.ipynb`, sofern der **Sage**-Kernel registriert ist. Für `nbconvert --execute` den Sage-Kernel angeben, z. B. `--ExecutePreprocessor.kernel_name=sagemath-10.9`.


## Imports

In [ ]:
from sage.all import RDF, gcd, Integer, is_prime, matrix
import math


## Funktion `calculate_rho_spec`

In [ ]:
def calculate_rho_spec(M, q, R, beta_E, beta_S):
    """
    Berechnet rho_spec(M) aus fuenf Faktoren (siehe Markdown-Einleitung).

    H_q(R): r != 0, |r| <= R, gcd(|r|, q) == 1.
    P_M: r in H_q mit is_prime(M + r).
    """
    M = Integer(M)
    q = Integer(q)
    R = int(R)

    H_q = [r for r in range(-R, R + 1) if r != 0 and gcd(abs(r), q) == 1]
    P_M = [r for r in H_q if is_prime(M + r)]
    K = len(P_M)
    if K == 0:
        return RDF(0)

    rho_slot = RDF(K) / RDF(len(H_q))

    def get_eabc_class(val):
        m = int(val) % 12
        if m == 1:
            return "E"
        if m == 5:
            return "A"
        if m == 7:
            return "B"
        if m == 11:
            return "C"
        return None

    classes = [get_eabc_class(M + r) for r in P_M]
    counts = {
        "E": classes.count("E"),
        "A": classes.count("A"),
        "B": classes.count("B"),
        "C": classes.count("C"),
    }
    ideal_n = RDF(K) / 4
    D_EABC = RDF(sum((RDF(counts[c]) - ideal_n) ** 2 for c in counts))
    B_EABC = RDF(math.exp(-float(beta_E) * float(D_EABC)))

    Pset = set(P_M)
    D_sym = sum(1 for r in P_M if -r not in Pset)
    d_sym = RDF(D_sym) / RDF(K)
    S_sym = RDF(math.exp(-float(beta_S) * float(d_sym)))

    P_M_sorted = sorted(P_M)
    class_seq = [get_eabc_class(M + r) for r in P_M_sorted]
    idx = {"E": 0, "A": 1, "B": 2, "C": 3}
    K_mat = matrix(RDF, 4, 4, 0)
    for i in range(len(class_seq) - 1):
        c1, c2 = class_seq[i], class_seq[i + 1]
        if c1 is not None and c2 is not None:
            K_mat[idx[c1], idx[c2]] += 1

    sigma = K_mat.singular_values()
    sum_sq = RDF(sum(s**2 for s in sigma))
    if sum_sq == 0:
        return RDF(0)

    lambdas = [RDF(s**2) / sum_sq for s in sigma]
    lambda_1 = max(lambdas)
    Theta_rank = RDF(1) - lambda_1

    S_ent = RDF(-sum(float(l) * math.log(float(l)) for l in lambdas if float(l) > 0))
    eta_ent = RDF(S_ent) / RDF(math.log(4))

    rho_spec = rho_slot * B_EABC * S_sym * Theta_rank * eta_ent
    return rho_spec


## Parameter und Beispiel-Kandidaten

`M_candidates` war im Rohfragment offen (`M_candidates =`). Hier eine **Startliste**; du kannst sie durch einen eigenen Scan ersetzen.

In [ ]:
q_val = 30
R_val = 29
beta_E_val = 1.0
beta_S_val = 1.0

M_candidates = [113160, 179070, 55818, 198450, 109494]

results = []
for M in M_candidates:
    rho = calculate_rho_spec(M, q_val, R_val, beta_E_val, beta_S_val)
    results.append((M, rho))
    print(f"M = {M}: rho_spec = {float(rho):.8f}")

best = max(results, key=lambda t: float(t[1]))
print("Maximum in dieser Liste:", best[0], float(best[1]))


M = 113160: rho_spec = 0.35250992
M = 179070: rho_spec = 0.04920545
M = 55818: rho_spec = 0.03609379
M = 198450: rho_spec = 0.00619910
M = 109494: rho_spec = 0.00033056
Maximum in dieser Liste: 113160 0.3525099192519932


## Optional: Scan über Vielfache von $q$

Schleife über zusammengesetzte $M \equiv 0 \pmod q$. Großes `scan_limit` kann lange laufen.

In [ ]:
def scan_primordial_multiples(q, R, beta_E, beta_S, scan_limit=50_000, step=None):
    step = step or q
    best_M, best_rho = None, RDF(-1)
    for M in range(q, scan_limit + 1, step):
        if is_prime(M):
            continue
        rho = calculate_rho_spec(M, q, R, beta_E, beta_S)
        if rho > best_rho:
            best_M, best_rho = M, rho
    return best_M, best_rho

# Demo auskommentiert — Limit nach Bedarf erhoehen
# best_M, best_rho = scan_primordial_multiples(q_val, R_val, beta_E_val, beta_S_val, scan_limit=200_000)
# print("Maximum:", best_M, float(best_rho))
